<div class="doris-cover">
  <div class="doris-cover-kicker">DEMO 05 · DBT × APACHE DORIS</div>
  <div class="doris-cover-title">Customer Snapshot</div>
  <p class="doris-cover-lead">Track customer attribute updates and hard deletes, then produce SCD Type 2 history and a current customer dimension.</p>
  <span class="doris-cover-note">Snapshot · check strategy · Hard Delete · SCD Type 2 · ref()</span>
</div>

## 1. Check the execution environment

Run this cell first. It uses the Demo dbt environment and current Doris connection settings, then confirms that a Backend is available.

In [ ]:
import importlib.util
from pathlib import Path


def find_demo_dir(start):
    for candidate in (start, *start.parents):
        demo_dir = candidate / "examples/doris-demos"
        if demo_dir.is_dir():
            return demo_dir
    raise FileNotFoundError("Start Jupyter from the dbt-for-apache-doris repository or a subdirectory.")


demo_dir = find_demo_dir(Path.cwd().resolve())
helper_path = demo_dir / "scripts/notebook_helpers.py"
helper_spec = importlib.util.spec_from_file_location("dbt_doris_notebook_helpers", helper_path)
notebook_helpers = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(notebook_helpers)

runner = notebook_helpers.DemoRunner()
runner.show_environment()

## 2. Demo 5: Customer Snapshot

CRM systems store the current customer state, while analytics and audit teams also need to know what attributes were in the past, when they changed, and what the last state was before deletion. This Demo delivers both customer history and a current dimension.

<table class="doris-index">
  <tr><th>Business users</th><td>CRM operations, customer analytics, and data governance teams</td></tr>
  <tr><th>Business question</th><td>How did customer email and type change, which records are current, and how do we retain deleted customer history?</td></tr>
  <tr><th>History rule</th><td><code>customer_id</code> is the business key; monitor selected attributes; hard deletes close the current version</td></tr>
  <tr><th>Delivered datasets</th><td>SCD Type 2 history <code>customer_snapshot</code>; current dimension <code>dim_customer_current</code></td></tr>
</table>

The cells below show two Snapshot rounds: record the initial state, then process an attribute update and a source-row deletion.

<div class="doris-flow">
  <div class="doris-flow-step"><strong>Current customers</strong>2 source rows</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>Staging View</strong><code>stg_customers</code></div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>First Snapshot</strong>2 valid history rows</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>Source changes</strong>Update customer 1, delete customer 2</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>SCD Type 2</strong>3 history rows, 1 current customer</div>
</div>

### 2.1 Prepare and inspect the current customer source

The source table uses a Unique Key to store the current customer state and initially contains Alice and Bob.

In [ ]:
snapshot_dir = runner.examples_root / "doris-customer-snapshot"
runner.show_file("Fixture SQL", snapshot_dir / "scripts/setup.sql")
runner.show_file("Source declarations", snapshot_dir / "models/sources.yml")
runner.run_sql_file("Create customer source table", snapshot_dir / "scripts/setup.sql")
runner.query("Input: current customer state", """
select customer_id, customer_number, customer_type, email
from dbt_demo_snapshot_source.CUSTOMERS
order by customer_id
""")

### 2.2 Create the staging View

`stg_customers` reads the current customer table with `source()` and provides a stable upstream node for the Snapshot.

In [ ]:
runner.show_file("Customer staging model", snapshot_dir / "models/stg_customers.sql")
runner.run_dbt("Create stg_customers View", snapshot_dir, "run", "--select", "stg_customers")
runner.query("Intermediate result: staging customers", """
select customer_id, customer_type, email
from dbt_demo_snapshot.stg_customers
order by customer_id
""")

### 2.3 Run the first Snapshot

The Snapshot uses the `check` strategy to monitor email, customer type, and other fields, with `invalidate_hard_deletes` enabled. The first run creates one valid version for each customer.

In [ ]:
runner.show_file("Snapshot definition", snapshot_dir / "snapshots/customer_snapshot.sql")
runner.run_dbt("Record initial customer versions", snapshot_dir, "snapshot", "--select", "customer_snapshot", "--threads", "1")
runner.query("First Snapshot: 2 valid history rows", """
select customer_id, email, dbt_valid_from, dbt_valid_to
from dbt_demo_snapshot_history.customer_snapshot
order by customer_id, dbt_valid_from
""")

### 2.4 Build the current customer dimension from the Snapshot

The dimension keeps only current versions where `dbt_valid_to is null` and counts the history versions for each customer. After the first build, both customers have one version.

In [ ]:
runner.show_file("Current customer dimension model", snapshot_dir / "models/dim_customer_current.sql")
runner.show_file("Data Test definition", snapshot_dir / "models/snapshot.yml")
runner.run_dbt("Create the current customer dimension", snapshot_dir, "run", "--select", "dim_customer_current")
runner.run_dbt("Test the current customer dimension", snapshot_dir, "test", "--select", "dim_customer_current", "--threads", "1")
runner.query("First dimension result", """
select customer_id, email, total_versions, has_history
from dbt_demo_snapshot.dim_customer_current
order by customer_id
""")

### 2.5 Update and delete source customers

Update customer 1's email and type, then delete customer 2 from the current source table. The Snapshot history is unchanged at this point.

In [ ]:
customer_changes_sql = """
update dbt_demo_snapshot_source.CUSTOMERS
set email = 'alice.new@example.com', customer_type = 'INDIVIDUAL_PLUS'
where customer_id = 1;
delete from dbt_demo_snapshot_source.CUSTOMERS where customer_id = 2
"""
runner.show_sql("Customer change SQL", customer_changes_sql)
runner.run_sql("Update Alice and delete Bob", customer_changes_sql)
runner.query("Changed current source", """
select customer_id, customer_type, email
from dbt_demo_snapshot_source.CUSTOMERS
order by customer_id
""")

### 2.6 Run the second Snapshot and refresh the dimension

The second Snapshot closes Alice's old version and creates a new one, while also closing Bob's version after the source deletion. The refreshed dimension contains only Alice as current.

In [ ]:
runner.run_dbt("Record customer changes", snapshot_dir, "snapshot", "--select", "customer_snapshot", "--threads", "1")
runner.run_dbt("Refresh the current customer dimension", snapshot_dir, "run", "--select", "dim_customer_current")
runner.run_dbt("Test the refreshed customer dimension", snapshot_dir, "test", "--select", "dim_customer_current", "--threads", "1")
runner.query("Second Snapshot: 3 history rows", """
select customer_id, email, dbt_valid_from, dbt_valid_to
from dbt_demo_snapshot_history.customer_snapshot
order by customer_id, dbt_valid_from
""")
runner.query("Refreshed current customer dimension", """
select customer_id, email, total_versions, has_history
from dbt_demo_snapshot.dim_customer_current
order by customer_id
""")

### 2.7 Verify SCD Type 2 history

The verifier checks that Alice has one closed version and one current version, Bob has only a closed version, and the current dimension contains only Alice.

In [ ]:
runner.run_script("Verify Snapshot Demo", snapshot_dir / "scripts/verify.sh")

## Complete

Both Snapshot rounds, customer history versions, hard-delete handling, and the current customer dimension passed verification.